# Plot Bionano_dataset

In [ ]:
import os
import cv2
import matplotlib.pyplot as plt
import numpy as np

def load_image_and_label(image_path, label_path):
    """
    Load an image and its corresponding label
    """
    # Load image
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)  # Convert BGR to RGB
    
    # Get image dimensions
    height, width, _ = image.shape
    
    # Load label
    with open(label_path, 'r') as f:
        label_content = f.read().strip()
    
    # Parse label (YOLO format: class_id center_x center_y width height)
    bboxes = []
    for line in label_content.split('\n'):
        parts = line.split()
        class_id = int(parts[0])
        # Convert normalized coordinates to pixel coordinates
        center_x = float(parts[1]) * width
        center_y = float(parts[2]) * height
        bbox_width = float(parts[3]) * width
        bbox_height = float(parts[4]) * height
        
        # Calculate top-left corner from center
        x1 = int(center_x - bbox_width / 2)
        y1 = int(center_y - bbox_height / 2)
        x2 = int(center_x + bbox_width / 2)
        y2 = int(center_y + bbox_height / 2)
        
        bboxes.append((class_id, x1, y1, x2, y2))
    
    return image, bboxes

def draw_bboxes(image, bboxes):
    """
    Draw bounding boxes on the image
    """
    image_with_boxes = image.copy()
    for bbox in bboxes:
        class_id, x1, y1, x2, y2 = bbox
        # Draw rectangle
        cv2.rectangle(image_with_boxes, (x1, y1), (x2, y2), (0, 255, 0), 2)
        # Add class label
        cv2.putText(image_with_boxes, f"Class {class_id}", (x1, y1-10), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
    
    return image_with_boxes

def main():
    # Define paths
    dataset_dir = "Dataset/bionano_cell"
    images_dir = os.path.join(dataset_dir, "images/train")
    labels_dir = os.path.join(dataset_dir, "labels/train")
    
    # Get the first image and label
    image_files = os.listdir(images_dir)
    if not image_files:
        print("No images found in the directory")
        return
    
    # Select the first image
    image_file = image_files[0]
    image_path = os.path.join(images_dir, image_file)
    
    # Get corresponding label file
    label_file = os.path.splitext(image_file)[0] + ".txt"
    label_path = os.path.join(labels_dir, label_file)
    
    # Check if label file exists
    if not os.path.exists(label_path):
        print(f"Label file not found: {label_path}")
        return
    
    # Load image and label
    image, bboxes = load_image_and_label(image_path, label_path)
    
    # Draw bounding boxes
    image_with_boxes = draw_bboxes(image, bboxes)
    
    # Display the image with bounding boxes
    plt.figure(figsize=(10, 8))
    plt.imshow(image_with_boxes)
    plt.title(f"Image: {image_file}")
    plt.axis('off')
    
    # Save the result
    output_path = "bionano_cell_with_label.png"
    plt.savefig(output_path, bbox_inches='tight', pad_inches=0.1)
    plt.close()
    
    print(f"Image with label saved to {output_path}")

if __name__ == "__main__":
    main()
